# Step 3b: Model Improvements

Two improvements over notebook 03:
1. **Logistic Regression** as alternative white-box (better regularisation than Decision Tree with small cohorts)
2. **Repeated GroupKFold (5×5)** — 25 evaluations instead of 5, much more stable AUC estimate

If results improve, we update the main conclusions. If not, we keep notebook 03 results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize, StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import joblib, os, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

classes  = pd.read_csv('../data/processed/class_names.csv').iloc[:, 0].tolist()
features = pd.read_csv('../models/selected_features.csv').iloc[:, 0].tolist()

# Load pre-selected 50 features
X      = np.load('../models/X_selected.npy')
y      = np.load('../models/y_all.npy')
groups = np.load('../data/processed/groups_all.npy')

print(f'X shape : {X.shape}  (50 ANOVA-selected features)')
print(f'Patients: {len(np.unique(groups))}')
print(f'Classes : {classes}')

## Improvement 1 — Repeated GroupKFold (5 repeats × 5 folds = 25 evaluations)

**Why?** With only 116 patients, a single 5-fold CV gives highly variable results (we saw AUC 0.57–0.84 across folds). Repeating the process 5 times with different random patient assignments averages out this variance.

Su et al. (2022) use 20 repeats × 5 folds. We use 5 repeats as a practical compromise.

In [ ]:
def evaluate_repeated_group_kfold(model, X, y, groups,
                                   n_splits=5, n_repeats=5,
                                   use_smote=True, model_name='Model'):
    """
    Repeated GroupKFold CV.
    Each repeat shuffles patients into folds differently.
    Returns mean ± std across all (n_splits * n_repeats) folds.
    """
    smote = SMOTE(random_state=42)
    all_f1, all_acc, all_auc = [], [], []
    all_y_true, all_y_pred, all_y_proba = [], [], []

    unique_groups = np.unique(groups)

    for repeat in range(n_repeats):
        # Shuffle patient order for this repeat
        rng = np.random.RandomState(repeat * 42)
        shuffled_groups = unique_groups.copy()
        rng.shuffle(shuffled_groups)

        # Reassign fold membership based on shuffled order
        group_to_fold = {g: i % n_splits
                         for i, g in enumerate(shuffled_groups)}
        fold_assignments = np.array([group_to_fold[g] for g in groups])

        for fold in range(n_splits):
            test_mask  = fold_assignments == fold
            train_mask = ~test_mask

            X_tr, X_te = X[train_mask], X[test_mask]
            y_tr, y_te = y[train_mask], y[test_mask]

            scaler  = StandardScaler()
            X_tr_sc = scaler.fit_transform(X_tr)
            X_te_sc = scaler.transform(X_te)

            if use_smote:
                try:
                    X_tr_sc, y_tr = smote.fit_resample(X_tr_sc, y_tr)
                except:
                    pass

            model.fit(X_tr_sc, y_tr)
            y_pred  = model.predict(X_te_sc)
            y_proba = model.predict_proba(X_te_sc)

            f1  = f1_score(y_te, y_pred, average='macro')
            acc = accuracy_score(y_te, y_pred)
            try:
                roc = roc_auc_score(y_te, y_proba,
                                    multi_class='ovr', average='macro')
            except:
                roc = np.nan

            all_f1.append(f1)
            all_acc.append(acc)
            all_auc.append(roc)
            all_y_true.extend(y_te)
            all_y_pred.extend(y_pred)
            all_y_proba.extend(y_proba)

        print(f'  Repeat {repeat+1}/5 done — '
              f'mean AUC so far: {np.nanmean(all_auc):.3f}')

    print(f'\n{model_name} — {n_repeats}×{n_splits} Repeated GroupKFold:')
    print(f'  Macro F1 : {np.mean(all_f1):.4f} ± {np.std(all_f1):.4f}')
    print(f'  Accuracy : {np.mean(all_acc):.4f} ± {np.std(all_acc):.4f}')
    print(f'  ROC-AUC  : {np.nanmean(all_auc):.4f} ± {np.nanstd(all_auc):.4f}')

    return {
        'all_f1':   all_f1,  'all_acc': all_acc,  'all_auc': all_auc,
        'y_true':   np.array(all_y_true),
        'y_pred':   np.array(all_y_pred),
        'y_proba':  np.array(all_y_proba),
        'mean_f1':  np.mean(all_f1),
        'mean_acc': np.mean(all_acc),
        'mean_auc': np.nanmean(all_auc)
    }

## XGBoost — Repeated GroupKFold

In [ ]:
xgb_best = joblib.load('../models/xgboost_best.pkl')

# Recreate with best params from notebook 03
xgb_rep = XGBClassifier(
    subsample=0.7, n_estimators=300, max_depth=5,
    learning_rate=0.1, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=42, n_jobs=-1
)

print('=== XGBoost — 5×5 Repeated GroupKFold ===')
xgb_rep_results = evaluate_repeated_group_kfold(
    xgb_rep, X, y, groups,
    n_splits=5, n_repeats=5,
    use_smote=True, model_name='XGBoost'
)

## Improvement 2 — Logistic Regression as White-Box

**Why Logistic Regression instead of Decision Tree?**
- L2 regularisation (C parameter) explicitly controls overfitting — important with small cohorts
- Produces calibrated probabilities → better ROC-AUC
- Coefficients are directly interpretable: each bacterial feature has a weight per class
- Used as white-box baseline in several microbiome classification papers

In [ ]:
lr_rep = LogisticRegression(
    C=0.1,           # strong regularisation for small cohort
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

print('=== Logistic Regression — 5×5 Repeated GroupKFold ===')
lr_rep_results = evaluate_repeated_group_kfold(
    lr_rep, X, y, groups,
    n_splits=5, n_repeats=5,
    use_smote=True, model_name='Logistic Regression'
)

## Also test Decision Tree with Repeated GroupKFold for fair comparison

In [ ]:
dt_rep = DecisionTreeClassifier(
    min_samples_split=5, min_samples_leaf=1,
    max_depth=15, criterion='entropy',
    class_weight='balanced', random_state=42
)

print('=== Decision Tree — 5×5 Repeated GroupKFold ===')
dt_rep_results = evaluate_repeated_group_kfold(
    dt_rep, X, y, groups,
    n_splits=5, n_repeats=5,
    use_smote=True, model_name='Decision Tree'
)

## Final Comparison — All Models

In [ ]:
comparison = pd.DataFrame({
    'Model': [
        'XGBoost (black-box)',
        'Logistic Regression (white-box)',
        'Decision Tree (white-box)'
    ],
    'Mean Accuracy ± SD': [
        f"{xgb_rep_results['mean_acc']:.3f} ± {np.std(xgb_rep_results['all_acc']):.3f}",
        f"{lr_rep_results['mean_acc']:.3f} ± {np.std(lr_rep_results['all_acc']):.3f}",
        f"{dt_rep_results['mean_acc']:.3f} ± {np.std(dt_rep_results['all_acc']):.3f}"
    ],
    'Mean Macro F1 ± SD': [
        f"{xgb_rep_results['mean_f1']:.3f} ± {np.std(xgb_rep_results['all_f1']):.3f}",
        f"{lr_rep_results['mean_f1']:.3f} ± {np.std(lr_rep_results['all_f1']):.3f}",
        f"{dt_rep_results['mean_f1']:.3f} ± {np.std(dt_rep_results['all_f1']):.3f}"
    ],
    'Mean ROC-AUC ± SD': [
        f"{xgb_rep_results['mean_auc']:.3f} ± {np.nanstd(xgb_rep_results['all_auc']):.3f}",
        f"{lr_rep_results['mean_auc']:.3f} ± {np.nanstd(lr_rep_results['all_auc']):.3f}",
        f"{dt_rep_results['mean_auc']:.3f} ± {np.nanstd(dt_rep_results['all_auc']):.3f}"
    ]
})
print(comparison.to_string(index=False))

In [ ]:
# Visualise AUC distribution across all 25 folds
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot of AUC across 25 folds
auc_data = pd.DataFrame({
    'XGBoost':              xgb_rep_results['all_auc'],
    'Logistic Regression':  lr_rep_results['all_auc'],
    'Decision Tree':        dt_rep_results['all_auc']
})
auc_data.boxplot(ax=axes[0], color='#5b8db8')
axes[0].axhline(0.5, color='red', linestyle='--',
                linewidth=1.5, label='Random (AUC=0.5)')
axes[0].set_ylabel('ROC-AUC (macro OvR)')
axes[0].set_title('AUC Distribution\n5×5 Repeated GroupKFold (25 folds)',
                  fontweight='bold')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=15)

# ROC curves XGBoost aggregated
y_bin = label_binarize(xgb_rep_results['y_true'], classes=[0, 1, 2])
colors_cls = ['#e07b54', '#5b8db8', '#6aab6a']
for i, (cls, col) in enumerate(zip(classes, colors_cls)):
    fpr, tpr, _ = roc_curve(y_bin[:, i],
                             xgb_rep_results['y_proba'][:, i])
    axes[1].plot(fpr, tpr, color=col, linewidth=2,
                 label=f'{cls} (AUC={auc(fpr,tpr):.3f})')
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('XGBoost ROC Curves\n(aggregated, 5×5 repeated GroupKFold)',
                  fontweight='bold')
axes[1].legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../figures/16_repeated_cv_comparison.png', bbox_inches='tight')
plt.show()

## Logistic Regression — Coefficient Interpretation (White-Box)

If Logistic Regression performs competitively, its coefficients give direct biological insight — positive coefficients mean higher abundance → more likely that class.

In [ ]:
# Train LR on full data for coefficient visualisation
scaler_full = StandardScaler()
X_full_sc   = scaler_full.fit_transform(X)
X_full_sm, y_full_sm = SMOTE(random_state=42).fit_resample(X_full_sc, y)

lr_final = LogisticRegression(
    C=0.1, class_weight='balanced',
    max_iter=1000, random_state=42
)
lr_final.fit(X_full_sm, y_full_sm)

# Plot top coefficients per class
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors_bar = ['#e07b54', '#6aab6a', '#5b8db8']

for ax, (cls_idx, cls_name, col) in zip(axes, [
    (0, classes[0], colors_bar[0]),
    (1, classes[1], colors_bar[1]),
    (2, classes[2], colors_bar[2])
]):
    coefs = lr_final.coef_[cls_idx]
    top_idx = np.argsort(np.abs(coefs))[::-1][:15]
    top_features = [features[i] for i in top_idx]
    top_coefs    = coefs[top_idx]

    bar_colors = [col if c > 0 else '#cccccc' for c in top_coefs]
    ax.barh(top_features[::-1], top_coefs[::-1],
            color=bar_colors[::-1], edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{cls_name}\nTop 15 features by |coefficient|',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Coefficient')

plt.suptitle('Logistic Regression Coefficients — White-Box Interpretation\n'
             'Positive = higher abundance → more likely this class',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/17_lr_coefficients.png', bbox_inches='tight')
plt.show()

## Decision — Which White-Box to Use?

Compare the final AUCs and decide here.

In [ ]:
print('FINAL DECISION:')
print(f'  XGBoost AUC          : {xgb_rep_results["mean_auc"]:.4f}')
print(f'  Logistic Reg AUC     : {lr_rep_results["mean_auc"]:.4f}')
print(f'  Decision Tree AUC    : {dt_rep_results["mean_auc"]:.4f}')
print()

if lr_rep_results['mean_auc'] > dt_rep_results['mean_auc']:
    print('→ Logistic Regression outperforms Decision Tree.')
    print('  Use LR as white-box in the final report and XAI notebook.')
    print('  Coefficients already provide direct biological interpretation.')
else:
    print('→ Decision Tree matches or outperforms Logistic Regression.')
    print('  Keep Decision Tree as white-box — tree structure is more intuitive.')

# Save best white-box for XAI
os.makedirs('../models', exist_ok=True)
joblib.dump(lr_final, '../models/logistic_regression_final.pkl')
joblib.dump(scaler_full, '../models/scaler_full.pkl')
np.save('../models/X_full_sm.npy', X_full_sm)
np.save('../models/y_full_sm.npy', y_full_sm)
print('\nModels saved for XAI notebook.')